# Install Dependencies

In [1]:
!pip install pandas numpy scikit-learn joblib python-docx torch --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 18.4 MB/s eta 0:00:00


# Imports

In [2]:
import os
import gc
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from collections import Counter
import joblib
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from docx import Document
from datetime import datetime
import json

# Mount Google Drive

In [3]:
from google.colab import drive
drive.mount('/users/')

Mounted at /users/


# Paths

In [4]:
RAW_BASE = "/users/"
BASE_SAVE_DIR = "/users/"

DATASET_NAME = "CIC_IoT_2023"
SAVE_DIR = os.path.join(BASE_SAVE_DIR, DATASET_NAME)
os.makedirs(SAVE_DIR, exist_ok=True)

print("RAW BASE :", RAW_BASE)
print("SAVE DIR :", SAVE_DIR)

RAW BASE : /users/
SAVE DIR : /users/


# Load CIC-IoT-2023

In [5]:
dfs = []

def load_file(path, fname):
    """
    Load supported files (CSV only for now).
    Low-memory mode enabled for safety.
    """
    if fname.lower().endswith(".csv"):
        print("Loading CSV:", fname)
        return pd.read_csv(path, low_memory=True)
    else:
        return None

for root, dirs, files in os.walk(RAW_BASE):
    for fname in files:
        fpath = os.path.join(root, fname)
        df_temp = load_file(fpath, fname)
        if df_temp is not None:
            dfs.append(df_temp)

if len(dfs) == 0:
    raise ValueError("❌ No supported data files found in CIC-IoT-2023 folder.")

df = pd.concat(dfs, ignore_index=True)
del dfs
gc.collect()

original_shape = df.shape
print("\n=== MERGED CIC-IoT-2023 DATASET ===")
print("Shape:", original_shape)
print("Preview columns:", list(df.columns)[:20])
df.head()

Loading CSV: Backdoor_Malware.pcap.csv
Loading CSV: BenignTraffic.pcap.csv
Loading CSV: BenignTraffic1.pcap.csv
Loading CSV: BenignTraffic2.pcap.csv
Loading CSV: BenignTraffic3.pcap.csv
Loading CSV: BrowserHijacking.pcap.csv
Loading CSV: CommandInjection.pcap.csv
Loading CSV: DDoS-ACK_Fragmentation.pcap.csv
Loading CSV: DDoS-ACK_Fragmentation1.pcap.csv
Loading CSV: DDoS-ACK_Fragmentation2.pcap.csv
Loading CSV: DDoS-ACK_Fragmentation3.pcap.csv
Loading CSV: DDoS-ACK_Fragmentation4.pcap.csv
Loading CSV: DDoS-ACK_Fragmentation5.pcap.csv
Loading CSV: DDoS-ACK_Fragmentation6.pcap.csv
Loading CSV: DDoS-ACK_Fragmentation7.pcap.csv
Loading CSV: DDoS-ACK_Fragmentation8.pcap.csv
Loading CSV: DDoS-ACK_Fragmentation9.pcap.csv
Loading CSV: DDoS-ACK_Fragmentation10.pcap.csv
Loading CSV: DDoS-ACK_Fragmentation11.pcap.csv
Loading CSV: DDoS-ACK_Fragmentation12.pcap.csv
Loading CSV: DDoS-HTTP_Flood-.pcap.csv
Loading CSV: DDoS-ICMP_Flood.pcap.csv
Loading CSV: DDoS-ICMP_Flood1.pcap.csv
Loading CSV: DDoS-IC

,Header_Length,Protocol Type,Time_To_Live,Rate,fin_flag_number,syn_flag_number,rst_flag_number,psh_flag_number,ack_flag_number,ece_flag_number,...,Tot sum,Min,Max,AVG,Std,Tot size,IAT,Number,Variance,Label
0,13.2,17,111.8,21.654112,0.0,0.0,0.0,0.0,0.3,0.0,...,2105.0,60.0,1392.0,210.5,415.549502,210.5,0.046181,10.0,172681.388889,NaN
1,11.2,17,63.5,134.621809,0.0,0.1,0.0,0.0,0.0,0.0,...,4736.0,60.0,1392.0,473.6,632.696206,473.6,0.008186,10.0,400304.488889,NaN
2,13.6,17,65.6,211.662495,0.0,0.1,0.0,0.0,0.2,0.0,...,3788.0,62.0,1392.0,378.8,508.763381,378.8,0.004735,10.0,258840.177778,NaN
3,24.8,6,84.4,155.707333,0.0,0.0,0.0,0.4,0.7,0.0,...,2917.0,62.0,833.0,291.7,308.737951,291.7,0.006422,10.0,95319.122222,NaN
4,10.4,17,118.3,105.440687,0.0,0.0,0.0,0.0,0.1,0.0,...,1163.0,62.0,230.0,116.3,75.052648,116.3,0.009484,10.0,5632.900000,NaN


# LABEL COLUMN NORMALIZATION

In [6]:
label_candidates = [
    "Label", "label",
    "Attack", "Attack_type", "Attack_Type",
    "Category", "Class", "class",
    "Attack_category", "Threat"
]

label_col = None
for col in df.columns:
    if col in label_candidates:
        label_col = col
        break

if label_col is None:
    raise ValueError("❌ No valid label column found in CIC-IoT-2023!")

print("✔ Using label column:", label_col)

df.rename(columns={label_col: "Label"}, inplace=True)

# Clean label values & drop missing
df = df[df["Label"].notna()].copy()
df["Label"] = df["Label"].astype(str).str.strip()

print("\nUnique raw labels (first 50):")
print(df["Label"].unique()[:50])
print("Total unique labels (raw):", df["Label"].nunique())

✔ Using label column: Label

Unique raw labels (first 50):
['DDOS-PSHACK_FLOOD' 'MIRAI-GREIP_FLOOD' 'DOS-UDP_FLOOD' 'DNS_SPOOFING'
 'DDOS-ICMP_FLOOD' 'DDOS-TCP_FLOOD' 'DDOS-SYN_FLOOD' 'DDOS-UDP_FLOOD'
 'MITM-ARPSPOOFING' 'DDOS-SYNONYMOUSIP_FLOOD' 'DOS-TCP_FLOOD'
 'VULNERABILITYSCAN' 'DOS-SYN_FLOOD' 'DDOS-RSTFINFLOOD' 'BENIGN'
 'DDOS-SLOWLORIS' 'DDOS-ICMP_FRAGMENTATION' 'MIRAI-GREETH_FLOOD'
 'RECON-HOSTDISCOVERY' 'MIRAI-UDPPLAIN' 'RECON-PORTSCAN'
 'DDOS-ACK_FRAGMENTATION' 'DDOS-UDP_FRAGMENTATION' 'RECON-OSSCAN'
 'BACKDOOR_MALWARE' 'DOS-HTTP_FLOOD' 'XSS' 'DDOS-HTTP_FLOOD'
 'BROWSERHIJACKING' 'SQLINJECTION' 'DICTIONARYBRUTEFORCE'
 'COMMANDINJECTION' 'RECON-PINGSWEEP' 'UPLOADING_ATTACK']
Total unique labels (raw): 34


# LABEL ENCODING

In [7]:
label_encoder = LabelEncoder()
df["Label_encoded"] = label_encoder.fit_transform(df["Label"])

classes = list(label_encoder.classes_)
num_classes = len(classes)

joblib.dump(label_encoder, os.path.join(SAVE_DIR, "label_encoder.pkl"))

print("\n✔ Encoded classes:")
for idx, name in enumerate(classes):
    print(f"  {idx}: {name}")
print("Total classes:", num_classes)


✔ Encoded classes:
  0: BACKDOOR_MALWARE
  1: BENIGN
  2: BROWSERHIJACKING
  3: COMMANDINJECTION
  4: DDOS-ACK_FRAGMENTATION
  5: DDOS-HTTP_FLOOD
  6: DDOS-ICMP_FLOOD
  7: DDOS-ICMP_FRAGMENTATION
  8: DDOS-PSHACK_FLOOD
  9: DDOS-RSTFINFLOOD
  10: DDOS-SLOWLORIS
  11: DDOS-SYNONYMOUSIP_FLOOD
  12: DDOS-SYN_FLOOD
  13: DDOS-TCP_FLOOD
  14: DDOS-UDP_FLOOD
  15: DDOS-UDP_FRAGMENTATION
  16: DICTIONARYBRUTEFORCE
  17: DNS_SPOOFING
  18: DOS-HTTP_FLOOD
  19: DOS-SYN_FLOOD
  20: DOS-TCP_FLOOD
  21: DOS-UDP_FLOOD
  22: MIRAI-GREETH_FLOOD
  23: MIRAI-GREIP_FLOOD
  24: MIRAI-UDPPLAIN
  25: MITM-ARPSPOOFING
  26: RECON-HOSTDISCOVERY
  27: RECON-OSSCAN
  28: RECON-PINGSWEEP
  29: RECON-PORTSCAN
  30: SQLINJECTION
  31: UPLOADING_ATTACK
  32: VULNERABILITYSCAN
  33: XSS
Total classes: 34


# FEATURE SELECTION (NUMERIC ONLY)

In [8]:
drop_cols = ["Label", "Label_encoded"]
# 'Attack' or similar columns may or may not exist, handle safely
for extra_col in ["Attack", "Attack_type", "Attack_Type", "Category", "Class", "class", "Attack_category", "Threat"]:
    if extra_col in df.columns and extra_col not in drop_cols:
        drop_cols.append(extra_col)

X_raw = df.drop(columns=drop_cols)
y = df["Label_encoded"].values

all_features = list(X_raw.columns)
original_num_features = len(all_features)

print("\n=== ORIGINAL FEATURE INFO ===")
print("Total features (before numeric filter):", original_num_features)

# Keep only numeric columns
numeric_cols = X_raw.select_dtypes(include=["number"]).columns.tolist()
X_num = X_raw[numeric_cols].copy()

del X_raw
gc.collect()

print("Numeric features kept:", len(numeric_cols))


=== ORIGINAL FEATURE INFO ===
Total features (before numeric filter): 39
Numeric features kept: 39


# CLEAN NUMERIC FEATURES & SCALE

In [9]:
X_num = X_num.replace([np.inf, -np.inf], np.nan)
X_num = X_num.fillna(X_num.median(numeric_only=True))

# Downcast to float32 to save RAM
X_num = X_num.astype("float32")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_num.values)

joblib_path = os.path.join(SAVE_DIR, "scaler.pkl")
joblib.dump(scaler, joblib_path)

print("\nScaler saved:", joblib_path)
print("Scaled shape:", X_scaled.shape)

# Free large intermediate frame
del X_num
gc.collect()


Scaler saved: /users/
Scaled shape: (45019234, 39)


0

# TRAIN/VAL/TEST SPLIT (70 / 15 / 15)

In [10]:
TEST_SIZE = 0.30
VAL_RATIO = 0.50

X_train, X_temp, y_train, y_temp = train_test_split(
    X_scaled, y, test_size=TEST_SIZE, stratify=y, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=VAL_RATIO, stratify=y_temp, random_state=42
)

print("\n=== SPLIT SHAPES ===")
print("Train:", X_train.shape, y_train.shape)
print("Val  :", X_val.shape,   y_val.shape)
print("Test :", X_test.shape,  y_test.shape)

# Class distributions
def get_class_distribution(y_arr):
    c = Counter(y_arr)
    return dict(sorted(c.items(), key=lambda x: x[0]))

train_dist = get_class_distribution(y_train)
val_dist   = get_class_distribution(y_val)
test_dist  = get_class_distribution(y_test)

print("\n=== CLASS DISTRIBUTIONS (ENCODED LABELS) ===")
print("Train:", train_dist)
print("Val  :", val_dist)
print("Test :", test_dist)


=== SPLIT SHAPES ===
Train: (31513463, 39) (31513463,)
Val  : (6752885, 39) (6752885,)
Test : (6752886, 39) (6752886,)

=== CLASS DISTRIBUTIONS (ENCODED LABELS) ===
Train: {np.int64(0): 2155, np.int64(1): 735961, np.int64(2): 3941, np.int64(3): 3618, np.int64(4): 190955, np.int64(5): 19318, np.int64(6): 4825281, np.int64(7): 303210, np.int64(8): 2744260, np.int64(9): 2710966, np.int64(10): 15680, np.int64(11): 2411961, np.int64(12): 2720291, np.int64(13): 3014260, np.int64(14): 3626719, np.int64(15): 192436, np.int64(16): 8765, np.int64(17): 120028, np.int64(18): 48159, np.int64(19): 1359523, np.int64(20): 1790779, np.int64(21): 2224126, np.int64(22): 664567, np.int64(23): 503759, np.int64(24): 596886, np.int64(25): 206128, np.int64(26): 90074, np.int64(27): 65779, np.int64(28): 1513, np.int64(29): 55111, np.int64(30): 3515, np.int64(31): 837, np.int64(32): 250308, np.int64(33): 2594}
Val  : {np.int64(0): 462, np.int64(1): 157706, np.int64(2): 845, np.int64(3): 775, np.int64(4): 40919

# AUTOENCODER MODEL (LATENT FEATURES)


In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nUsing device:", device)

input_dim = X_train.shape[1]
latent_dim = 64  # same as other datasets

class Autoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        out = self.decoder(z)
        return out, z


Using device: cuda


In [12]:
ae_model = Autoencoder(input_dim=input_dim, latent_dim=latent_dim).to(device)
optimizer = torch.optim.Adam(ae_model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

In [13]:
train_tensor = torch.tensor(X_train, dtype=torch.float32)
train_dataset = TensorDataset(train_tensor)
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)

In [14]:
EPOCHS = 10
epoch_losses = []

print("\n=== AUTOENCODER TRAINING START ===")
for epoch in range(1, EPOCHS + 1):
    ae_model.train()
    running_loss = 0.0
    total_samples = 0

    for (batch_x,) in train_loader:
        batch_x = batch_x.to(device)
        optimizer.zero_grad()
        reconstructed, z = ae_model(batch_x)
        loss = criterion(reconstructed, batch_x)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * batch_x.size(0)
        total_samples += batch_x.size(0)

    avg_loss = running_loss / total_samples
    epoch_losses.append(avg_loss)
    print(f"Epoch {epoch}/{EPOCHS} - Loss: {avg_loss:.6f}")

print("=== AUTOENCODER TRAINING DONE ===")


=== AUTOENCODER TRAINING START ===
Epoch 1/10 - Loss: 0.452901
Epoch 2/10 - Loss: 0.060171
Epoch 3/10 - Loss: 0.036638
Epoch 4/10 - Loss: 0.040726
Epoch 5/10 - Loss: 0.019272
Epoch 6/10 - Loss: 0.043437
Epoch 7/10 - Loss: 0.031914
Epoch 8/10 - Loss: 0.035441
Epoch 9/10 - Loss: 0.114001
Epoch 10/10 - Loss: 0.009718
=== AUTOENCODER TRAINING DONE ===


# LATENT EMBEDDING EXTRACTION

In [15]:
def get_latent_embeddings(model, X_array, batch_size=4096):
    model.eval()
    all_z = []
    with torch.no_grad():
        for i in range(0, len(X_array), batch_size):
            batch = torch.tensor(X_array[i:i+batch_size], dtype=torch.float32).to(device)
            _, z = model(batch)
            all_z.append(z.cpu().numpy())
    return np.vstack(all_z)

print("\n🔁 Generating latent embeddings...")

z_train = get_latent_embeddings(ae_model, X_train)
z_val   = get_latent_embeddings(ae_model, X_val)
z_test  = get_latent_embeddings(ae_model, X_test)

print("Latent shapes:")
print("Train:", z_train.shape)
print("Val  :", z_val.shape)
print("Test :", z_test.shape)


🔁 Generating latent embeddings...
Latent shapes:
Train: (31513463, 64)
Val  : (6752885, 64)
Test : (6752886, 64)


# SAVE LATENT SPLITS + LABELS + FEATURES

In [16]:
np.save(os.path.join(SAVE_DIR, "train_latent.npy"), z_train)
np.save(os.path.join(SAVE_DIR, "val_latent.npy"),   z_val)
np.save(os.path.join(SAVE_DIR, "test_latent.npy"),  z_test)

np.save(os.path.join(SAVE_DIR, "y_train.npy"), y_train)
np.save(os.path.join(SAVE_DIR, "y_val.npy"),   y_val)
np.save(os.path.join(SAVE_DIR, "y_test.npy"),  y_test)

# save AE
torch.save(ae_model.state_dict(), os.path.join(SAVE_DIR, "autoencoder.pth"))

# save feature names (numeric ones used for training)
feature_list_path = os.path.join(SAVE_DIR, "feature_list.txt")
with open(feature_list_path, "w") as f:
    for col in numeric_cols:
        f.write(col + "\n")

print("\n✔ Saved:")
print("  - Latent splits (train/val/test)")
print("  - Label splits (train/val/test)")
print("  - Scaler")
print("  - Label encoder")
print("  - Autoencoder weights")
print("  - Feature list:", feature_list_path)


✔ Saved:
  - Latent splits (train/val/test)
  - Label splits (train/val/test)
  - Scaler
  - Label encoder
  - Autoencoder weights
  - Feature list: /users/


# JSON SUMMARY

In [17]:
def convert_keys_to_int(d):
    return {int(k): int(v) for k, v in d.items()}

train_dist = convert_keys_to_int(train_dist)
val_dist   = convert_keys_to_int(val_dist)
test_dist  = convert_keys_to_int(test_dist)


label_index_to_name = {int(i): str(name) for i, name in enumerate(classes)}

summary = {
    "dataset_name": DATASET_NAME,
    "generated_at": datetime.now().isoformat(),

    "raw": {
        "num_samples": int(original_shape[0]),
        "num_features_total": int(original_shape[1]),
        "feature_names_total": all_features,
    },

    "numeric_features": {
        "num_numeric_features": len(numeric_cols),
        "feature_names_numeric": numeric_cols,
    },

    "classes": {
        "num_classes": num_classes,
        "index_to_name": label_index_to_name,
    },

    "splits": {
        "train": {
            "num_samples": int(X_train.shape[0]),
            "class_counts": train_dist,
        },
        "val": {
            "num_samples": int(X_val.shape[0]),
            "class_counts": val_dist,
        },
        "test": {
            "num_samples": int(X_test.shape[0]),
            "class_counts": test_dist,
        },
    },

    "latent": {
        "latent_dim": int(latent_dim),
    },
}

# Save JSON
summary_path = os.path.join(SAVE_DIR, "preprocessing_summary.json")
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=4)

print("💾 JSON summary saved to:", summary_path)

💾 JSON summary saved to: /users/


# DOCX SUMMARY

In [18]:
doc = Document()
doc.add_heading(f"Dataset Preprocessing Summary — {DATASET_NAME}", level=1)

doc.add_paragraph(f"Generated at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# ---------------------------------------------------------
# 1. RAW DATASET
# ---------------------------------------------------------
doc.add_heading("1. Raw Dataset Overview", level=2)

p = doc.add_paragraph()
p.add_run("Total samples: ").bold = True
p.add_run(str(original_shape[0]))

p = doc.add_paragraph()
p.add_run("Total features (before preprocessing): ").bold = True
p.add_run(str(original_shape[1]))

doc.add_paragraph("Feature names (before preprocessing):")
for name in all_features[:50]:
    doc.add_paragraph(f"- {name}", style="List Bullet")
if len(all_features) > 50:
    doc.add_paragraph(f"... and {len(all_features) - 50} more features.")

# ---------------------------------------------------------
# 2. NUMERIC FEATURES
# ---------------------------------------------------------
doc.add_heading("2. Numeric Feature Selection", level=2)

p = doc.add_paragraph()
p.add_run("Numeric features selected: ").bold = True
p.add_run(str(len(numeric_cols)))

doc.add_paragraph("Numeric feature names:")
for name in numeric_cols[:50]:
    doc.add_paragraph(f"- {name}", style="List Bullet")
if len(numeric_cols) > 50:
    doc.add_paragraph(f"... and {len(numeric_cols) - 50} more features.")

# ---------------------------------------------------------
# 3. CLASSES
# ---------------------------------------------------------
doc.add_heading("3. Classes", level=2)

p = doc.add_paragraph()
p.add_run("Total classes: ").bold = True
p.add_run(str(num_classes))

table = doc.add_table(rows=1, cols=2)
hdr = table.rows[0].cells
hdr[0].text = "Class Index"
hdr[1].text = "Class Name"

for idx, name in label_index_to_name.items():
    row = table.add_row().cells
    row[0].text = str(idx)
    row[1].text = str(name)

# ---------------------------------------------------------
# 4. SPLIT STATS
# ---------------------------------------------------------
doc.add_heading("4. Train / Validation / Test Split", level=2)

for split_name, stats in summary["splits"].items():
    doc.add_heading(f"{split_name.upper()} Split", level=3)

    p = doc.add_paragraph()
    p.add_run("Samples: ").bold = True
    p.add_run(str(stats["num_samples"]))

    doc.add_paragraph("Class counts:")
    items = list(stats["class_counts"].items())

    for k, v in items[:50]:
        doc.add_paragraph(f"Class {k}: {v}", style="List Bullet")

    if len(items) > 50:
        doc.add_paragraph(f"... and {len(items) - 50} more classes.", style="List Bullet")

# ---------------------------------------------------------
# 5. LATENT INFO
# ---------------------------------------------------------
doc.add_heading("5. Latent Representation", level=2)
p = doc.add_paragraph()
p.add_run("Latent dimension (autoencoder): ").bold = True
p.add_run(str(latent_dim))

# SAVE DOCX
docx_path = os.path.join(SAVE_DIR, "preprocessing_summary.docx")
doc.save(docx_path)

print("💾 DOCX summary saved to:", docx_path)


💾 DOCX summary saved to: /users/
